In [ ]:
pip install pandas scikit-learn

In [1]:
# ===== Step 0: imports =====
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# ===== Step 1: paths =====
noise_csv = r"D:\CHIRP\analysis\outputs\noise_features.csv"
bird_csv = r"D:\CHIRP\analysis\outputs\features_04_parameterTest.csv"

METADATA_COLS = [
    "Number", "FileName", "Segment",
    "Valence", "Arousal", "SocialEngagement", "Time",
    "Behavior", "Gender", "Age",
]

# ===== Step 2: load data and label (0 = noise, 1 = bird) =====
noise_df = pd.read_csv(noise_csv)
bird_df = pd.read_csv(bird_csv)

feature_cols = [c for c in noise_df.columns if c not in METADATA_COLS]
assert feature_cols == [c for c in bird_df.columns if c not in METADATA_COLS]

noise_df["Class"] = 0  # noise
bird_df["Class"] = 1   # bird

data = pd.concat(
    [noise_df[feature_cols + ["Class"]], bird_df[feature_cols + ["Class"]]],
    ignore_index=True,
)
print(f"Samples: {len(data)} (noise={len(noise_df)}, bird={len(bird_df)})")
print(f"Features: {len(feature_cols)}")

X = data[feature_cols]
y = data["Class"]

# ===== Step 3: train / test split =====
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

# ===== Step 4: random forest classifier (audio features only) =====
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# ===== Step 5: test metrics (MAE and R² on encoded labels) =====
y_pred = clf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- Test set results ---")
print(f"MAE: {mae:.4f}")
print(f"R²:  {r2:.4f}")

Samples: 450 (noise=134, bird=316)
Features: 41
Train: 360, Test: 90

--- Test set results ---
MAE: 0.0222
R²:  0.8942
